## Assignment 3: $k$ Nearest Neighbor


**Q1.**
1. What is the difference between regression and classification?
2. What is a confusion table? What does it help us understand about a model's performance?
3. What does the SSE quantify about a particular model?
4. What are overfitting and underfitting? 
5. Why does splitting the data into training and testing sets, and choosing $k$ by evaluating accuracy or SSE on the test set, improve model performance?
6. With classification, we can report a class label as a prediction or a probability distribution over class labels. Please explain the strengths and weaknesses of each approach.

**Q2.** This is a case study on $k$ nearest neighbor classification, using the `land_mines.csv` data.

The data consists of a label, `mine_type`, taking integer values 1 to 5, and three properties of the mine, `voltage`, `height` and `soil`. We want to predict the kind of mine from data about it. Imagine working for the DOD or a humanitarian aid agency, trying to help people remove land mines more safely.

1. Load the data. Perform some EDA, summarizing the target label and the features.
2. Split the sample 50/50 into training and test/validation sets. (The smaller the data are, the more equal the split should be, in my experience: Otherwise, all of the members of one class end up in the training or test data, and the model falls apart.)
3. Build a $k$-NN classifier. Explain how you select $k$.
4. Print a confusion table for the optimal model, comparing predicted and actual class label on the test set. How accurate is it? Where is performance more or less accurate?
5. Notice that you can have a lot of accurate predictions for a given type of mine, but still make a lot of mistakes. Please explain how you'd advise someone to actually use this predictive model in practice, given the errors that it tends to make.

---

## *Phase 1: Solution without AI

Complete all questions in the following cells without generative AI. Then commit and push the
notebook before beginning Phase 2.

**Declaration:** I completed this version without generative AI.

**Student(s):** Andrew Kim

**Date:** 10/2/26

In [71]:
# q1.1
# a regression predicts a numeric outcome such as price
# a classification predicts a categorical outcome such as type of product

# q1.2
# a confusion table counts predictions by their actual and predicted values
# it helps evaluate the accuracy of a classification model by showing the number of correct and incorrect predictions based on class

# q1.3
# the sum squared error (SSE) is a measure of the total deviation of a prediction from actual values
# it is found by summing the squares of differences between predicted and actual values
# it quantifies how accurate a regression model is, with lower sse being better

# q1.4
# overfitting occurs when a model is too broad and shows noise 
# small ks can cause overfitting leading to highly varied predictions
# underfitting occurs when a model is too narrow and does not capture patterns in data
# large ks can cause underfitting leading to almost constant predictions

# q1.5
# splitting the data into training and testing sets lets the model learn from one set and evalute its performance on an unseen set
# choosing k this way helps find a ideal k value that avoids under or overfitting the data

# q1.6
# using class labels can be easier to interpret and be better evaluated 
# but can hide uncertainity or important information about the data
# using probabilities can show the confidence and uncertaintly of a model
# but can be harder to interpret/evaluate or be poorly callibrated

In [72]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.neighbors import KNeighborsClassifier

In [73]:
# q2.1
df = pd.read_csv('data/land_mines.csv')
df.head()
print(df.shape)
print(df["mine_type"].value_counts().sort_index())
print(df[["voltage", "height", "soil"]].describe())
print(df.duplicated().sum())
print(df.isna().sum())

# data contains 338 rows with 4 columns (three predictors and one target)
# the spread of mine types are fairly even and there are no missing or duplicated values
# the predictor values scale from about 0 to 1

(338, 4)
mine_type
1    71
2    70
3    66
4    66
5    65
Name: count, dtype: int64
          voltage      height        soil
count  338.000000  338.000000  338.000000
mean     0.430634    0.508876    0.503550
std      0.195819    0.306043    0.344244
min      0.197734    0.000000    0.000000
25%      0.309737    0.272727    0.200000
50%      0.359516    0.545455    0.600000
75%      0.482628    0.727273    0.800000
max      0.999999    1.000000    1.000000
0
voltage      0
height       0
soil         0
mine_type    0
dtype: int64


In [74]:
# q2.2
x = df[["voltage", "height", "soil"]]
y = df["mine_type"]

x_train, x_valid, y_train, y_valid = train_test_split(x, y, test_size=0.5, random_state=42, stratify=y)

scaler = MinMaxScaler()
x_train_scaled = pd.DataFrame(scaler.fit_transform(x_train), columns=x_train.columns, index=x_train.index)
x_valid_scaled = pd.DataFrame(scaler.transform(x_valid), columns=x_valid.columns, index=x_valid.index)

model = KNeighborsClassifier(n_neighbors=3).fit(x_train_scaled, y_train)

In [75]:
# q2.3
ks = np.arange(1, 51)

valid_accuracy = []
train_accuracy = []

for k in ks:
    model = KNeighborsClassifier(n_neighbors=k)
    model.fit(x_train_scaled, y_train)
    valid_hat = model.predict(x_valid_scaled)
    train_hat = model.predict(x_train_scaled)
    valid_accuracy.append(np.mean(valid_hat == y_valid))
    train_accuracy.append(np.mean(train_hat == y_train))

k_star = int(ks[np.argmax(valid_accuracy)])
k_star

# because we are using a classification model, i used accuracy to evalute k.
# for looped k values 1-50 and selected the value with the highest accuracy as the k to use

1

In [ ]:
# q2.4
k_model = KNeighborsClassifier(n_neighbors=k_star)

k_model.fit(x_train_scaled, y_train)

y_pred = k_model.predict(x_valid_scaled)

confusion = pd.crosstab(y_valid, y_pred, rownames=['actual'], colnames=['predicted'])

print(confusion)
print('accuracy =', np.mean(y_pred == y_valid))

# the optimal model has an accuracy of about 50 percent
# type 2 mines had the higest accuracy of prediction with type 1 also being somewhat accurate
# type 3,4,5 had the lowest accuracy indicating the model is good for type 2 mines but not the others

predicted   1   2  3   4  5
actual                     
1          21   0  4   4  7
2           0  32  0   3  0
3           8   0  7  10  8
4           7   5  4  11  6
5           7   0  9   9  7
accuracy = 0.46153846153846156


In [ ]:
# q2.5
# the overall accuracy of the model being less than 50 percent is strong evidence as to not use this to predict mine type by itself
# the model could have use confirming mine types, specifically for mine 2, but considering the risk of mines, it would not be advisable to use this model for any other purpose than to potentially confirm observations
# although the model has a strong accuracy for type 2 mines, its overall accuracy brings down the use case for the model.

## *Phase 2: AI-assisted Revision
Keep Phase 1 frozen.

- **Phase 1 commit SHA ID:** [To be provided]
- **AI tool and model used:** [To be provided]

### *Prompts used

1. taking the following juypter notebook, suggest changes to the phase 1 section individually and as a summarized list, showing all suggested changes in accordance to the phase 2 guidelines

### *AI-proposed changes


[Paste the AI's concise numbered list verbatim.]

1. ...
2. ...

### *Evaluation of AI suggestions


| Change | Decision | Reason and verification |
|---|---|---|
| 1 | Accept / modify / reject | ... |
| 2 | Accept / modify / reject | ... |

### *AI-assisted solution in full
After the evaluation of AI suggestions, incorporate the accepted revisions into your Phase 1 solution and provide the final Phase 2 solution in full in the following cells.

In [78]:
# Your Phase 2 solution to the problem goes here.


### *Reflection on AI assistance
In your own words without generative AI.

[In 150–300 words, describe the main improvements, AI mistakes or limitations, how the final solution was verified, and any remaining concerns.]